# C

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
# brentq: find a root of a function
# minimize_scalar: minimize a scalar function of one variable
from scipy.optimize import brentq, minimize_scalar
# quad: compute a definite integral of a function
from scipy.integrate import quad

In [12]:
est = {"q0": 20, "p0": 6, "elasticity": -0.5, "passthrough": 0.9}

In [13]:
# calibrate: pick slopes so the curves match elasticity and passthrough
q0, p0 = est["q0"], est["p0"]
e, pt = est["elasticity"], est["passthrough"]

d1 = -p0 / (e * q0)        # from e = -(1/d1)(p0/q0)
s1 = d1 * (1 - pt) / pt    # from pt = d1/(d1+s1)

# inverse curves: price as a function of quantity
# demand: p = d0 - d1*q, supply: p = s0 + s1*q, both pass through (q0, p0)
d0 = p0 + d1 * q0
s0 = p0 - s1 * q0

def p_demand(q): return d0 - d1 * q     # buyers' price
def p_supply(q): return s0 + s1 * q     # sellers' price (before the new tax)

print(d1, s1, d0, s0)

0.6 0.06666666666666665 18.0 4.666666666666667


In [14]:
t = 2

# brentq: finds q where the gap between buyers' and sellers' price equals t
q_t = brentq(lambda q: p_demand(q) - p_supply(q) - t, 0, q0)
pb, ps = p_demand(q_t), p_supply(q_t)

# quad: area under a curve between two points
dCS = quad(lambda q: p_demand(q) - pb, 0, q_t)[0] - quad(lambda q: p_demand(q) - p0, 0, q0)[0]
dPS = quad(lambda q: ps - p_supply(q), 0, q_t)[0] - quad(lambda q: p0 - p_supply(q), 0, q0)[0]
rev = t * q_t
dwl = quad(lambda q: p_demand(q) - p_supply(q), q_t, q0)[0]

print(q_t, pb, ps, dCS, dPS, rev, dwl)
print("check (should be ~0):", dCS + dPS + rev + dwl)

17.0 7.800000000000001 5.8 -33.30000000000001 -3.700000000000003 34.0 3.000000000000001
check (should be ~0): -1.3322676295501878e-14


In [15]:
# minimize_scalar: finds the t that minimizes -revenue, i.e. maximizes revenue
def neg_rev(t):
    q = (d0 - s0 - t) / (d1 + s1)
    return -t * q

best = minimize_scalar(neg_rev, bounds=(0, d0 - s0), method="bounded")
print(best.x, (d0 - s0) / 2)

import os
os.makedirs("results", exist_ok=True)
out = dict(q=q_t, p_buyers=pb, p_sellers=ps, dCS=dCS, dPS=dPS, revenue=rev, dwl=dwl, best_tax=best.x)
json.dump(out, open("results/model.json", "w"))

6.666666666666666 6.666666666666666
